# SURENA VLA — Episode Evaluation (Single & Batch)

Single self-contained notebook: HDF5 extraction, statistics, figure generation and cross-preset numerical tables for `EpisodeLogger`'s `.h5` episode logs.

**Per task preset:** 
1. IK nested donut 
2. latency violins
3. IK position-error histogram
4. IK orientation-error histogram 
5. EEF trajectory tracking (representative episode)
6. joint trajectory tracking (7 subplots)
7. motion-smoothness distributions
8. episode-video exports (plain MP4 + action-history MP4)


## 1. Imports, palette, style, constants

In [1]:
from __future__ import annotations

import argparse
import glob
import json
import math
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Sequence

import h5py
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy import stats as scipy_stats

### Style, palette, constants

In [2]:
PAPER_RC = {
    "figure.dpi": 120,
    "savefig.dpi": 320,
    "font.size": 12,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "mathtext.fontset": "dejavuserif",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.color": ".8",
    "axes.edgecolor": "0.15",
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#cccccc",
}

# Tol muted / Okabe-Ito blend - colorblind safe, print friendly.
C_BLUE, C_RED, C_GREEN = "#4477AA", "#EE6677", "#228833"
C_YELLOW, C_CYAN, C_PURPLE, C_GREY = "#CCBB44", "#66CCEE", "#AA3377", "#505050"
C_ORANGE = "#F59F00"

IK_COLORS = {
    "full_pose": "#1B7837",
    "relaxed_orientation": "#7FBC70",
    "nearest_feasible": C_ORANGE,
    "hold_current": "#C92A2A",
    "other": "#9E9E9E",
    "success": "#2E7D32",
    "failure": "#B02418",
}
IK_LABELS = {
    "full_pose": "Full pose",
    "relaxed_orientation": "Relaxed orientation",
    "nearest_feasible": "Nearest feasible",
    "hold_current": "Hold current",
    "other": "Unclassified",
}
IK_SUCCESS_KEYS = ("full_pose", "relaxed_orientation", "nearest_feasible")
IK_FAILURE_KEYS = ("hold_current",)
IK_ALL_KEYS = IK_SUCCESS_KEYS + IK_FAILURE_KEYS + ("other",)

JOINT_NAMES = ["Shoulder pitch", "Shoulder roll", "Elbow", "Forearm roll",
               "Forearm link", "Hand pitch", "Hand roll"]

FIG_DONUT = (6.8, 4.8)
FIG_HIST = (7.2, 4.6)
FIG_VIOLIN_H = (7.6, 4.4)
FIG_SMOOTHNESS = (11.2, 4.4)
FIG_JOINTS = (13.6, 7.0)

SMOOTHNESS_START_S = 1.0     # discard startup transient (as in the reference notebook)
WARN_CLEARANCE_M = 0.02      # clearance below this counts as a "warning" collision event
MAX_VIOLIN_SAMPLES = 20000   # plotting decimation only; statistics always use all samples

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Statistics helpers

In [3]:
def ci95_factor(n: int) -> float:
    """t(0.975, df=n-1); NaN when undefined."""
    n = int(n)
    return float(scipy_stats.t.ppf(0.975, df=n - 1)) if n >= 2 else float("nan")


def mean_std_ci(vals: Iterable[float]) -> tuple[float, float, float, int]:
    """(mean, std ddof=1, half-width of the 95% CI, n) - NaN aware."""
    a = np.asarray(list(vals), dtype=float).ravel()
    a = a[np.isfinite(a)]
    n = a.size
    if n == 0:
        return float("nan"), float("nan"), float("nan"), 0
    mean = float(a.mean())
    if n == 1:
        return mean, float("nan"), float("nan"), 1
    std = float(a.std(ddof=1))
    ci = ci95_factor(n) * std / math.sqrt(n)
    return mean, std, float(ci), n


@dataclass
class Stat:
    """Pooled sample statistics + episode-level CI of the mean."""
    mean: float = float("nan")
    std: float = float("nan")
    ci: float = float("nan")
    lo: float = float("nan")
    hi: float = float("nan")
    n_samples: int = 0
    n_episodes: int = 0
    median: float = float("nan")
    p5: float = float("nan")
    p95: float = float("nan")

    def pm_std(self, prec: int = 2) -> str:
        if not np.isfinite(self.mean):
            return "n/a"
        if not np.isfinite(self.std):
            return f"{self.mean:.{prec}f}"
        return f"{self.mean:.{prec}f} +/- {self.std:.{prec}f}"

    def ci_str(self, prec: int = 2) -> str:
        if not (np.isfinite(self.lo) and np.isfinite(self.hi)):
            return "n/a"
        return f"[{self.lo:.{prec}f}, {self.hi:.{prec}f}]"

    def as_dict(self) -> dict[str, float]:
        return dict(mean=self.mean, std=self.std, ci95=self.ci, ci_lo=self.lo, ci_hi=self.hi,
                    median=self.median, p5=self.p5, p95=self.p95,
                    n_samples=self.n_samples, n_episodes=self.n_episodes)


def pooled_stat(per_episode_samples: Sequence[Any]) -> Stat:
    """Pooled mean/std over every sample; CI of the mean from the episode means."""
    arrays = [np.asarray(a, float).ravel() for a in per_episode_samples if a is not None]
    arrays = [a[np.isfinite(a)] for a in arrays]
    arrays = [a for a in arrays if a.size]
    if not arrays:
        return Stat()
    pooled = np.concatenate(arrays)
    ep_means = np.array([a.mean() for a in arrays], float)
    mean = float(pooled.mean())
    std = float(pooled.std(ddof=1)) if pooled.size > 1 else float("nan")
    if ep_means.size >= 2:                      # episode-level CI (respects clustering)
        ci = ci95_factor(ep_means.size) * ep_means.std(ddof=1) / math.sqrt(ep_means.size)
        centre = float(ep_means.mean())
    elif pooled.size > 1:                       # single episode: fall back to sample SEM
        ci = ci95_factor(pooled.size) * std / math.sqrt(pooled.size)
        centre = mean
    else:
        ci, centre = float("nan"), mean
    return Stat(mean=mean, std=std, ci=float(ci),
                lo=centre - ci if np.isfinite(ci) else float("nan"),
                hi=centre + ci if np.isfinite(ci) else float("nan"),
                n_samples=int(pooled.size), n_episodes=len(arrays),
                median=float(np.median(pooled)),
                p5=float(np.percentile(pooled, 5)), p95=float(np.percentile(pooled, 95)))


def episode_stat(values: Sequence[float]) -> Stat:
    """Statistics of one scalar per episode."""
    mean, std, ci, n = mean_std_ci(values)
    a = np.asarray(list(values), float)
    a = a[np.isfinite(a)]
    return Stat(mean=mean, std=std, ci=ci,
                lo=mean - ci if np.isfinite(ci) else float("nan"),
                hi=mean + ci if np.isfinite(ci) else float("nan"),
                n_samples=int(a.size), n_episodes=int(a.size),
                median=float(np.median(a)) if a.size else float("nan"),
                p5=float(np.percentile(a, 5)) if a.size else float("nan"),
                p95=float(np.percentile(a, 95)) if a.size else float("nan"))

### HDF5 access - schema aware, tolerant to missing signals

In [4]:
def _txt(v: Any) -> str:
    if isinstance(v, bytes):
        return v.decode(errors="replace")
    if isinstance(v, np.generic):
        v = v.item()
        return v.decode(errors="replace") if isinstance(v, bytes) else str(v)
    return str(v)


def _decode_text_array(values) -> np.ndarray:
    return np.array([_txt(x) for x in np.atleast_1d(values)], dtype=object)


def _get(f: h5py.File, *names: str):
    """First existing dataset among `names` (text decoded to an object array)."""
    for name in names:
        if name in f and isinstance(f[name], h5py.Dataset):
            data = f[name][()]
            if getattr(data, "dtype", None) is not None and data.dtype.kind in "SUO":
                return _decode_text_array(data)
            return np.asarray(data)
    return None


def _attr(f: h5py.File, *names: str, default=None):
    for name in names:
        if name in f.attrs:
            v = f.attrs[name]
            if isinstance(v, bytes):
                return v.decode(errors="replace")
            if isinstance(v, np.generic):
                return v.item()
            return v
    return default


def classify_ik_stage(name: str) -> str:
    """Map a logged IK stage string onto the four hierarchy levels of Figure A2."""
    s = str(name).strip().lower().replace("-", "_").replace(" ", "_")
    if any(k in s for k in ("hold", "fail", "infeasible", "none", "abort", "freeze", "reject")):
        return "hold_current"
    if "full" in s:
        return "full_pose"
    if any(k in s for k in ("relax", "pos_only", "position_only", "posonly", "orient",
                            "no_rot", "ignore_rot", "free_rot")):
        return "relaxed_orientation"
    if any(k in s for k in ("nearest", "feasible", "closest", "clamp", "partial", "best",
                            "limit", "projected")):
        return "nearest_feasible"
    return "other"


def quat_to_euler(q) -> np.ndarray:
    """wxyz quaternion -> roll/pitch/yaw [rad] (matches surena_vla.telemetry.codec)."""
    w, x, y, z = [float(v) for v in np.asarray(q, float).ravel()[:4]]
    roll = math.atan2(2 * (w * x + y * z), 1 - 2 * (x * x + y * y))
    sinp = 2 * (w * y - z * x)
    pitch = math.copysign(math.pi / 2, sinp) if abs(sinp) >= 1 else math.asin(sinp)
    yaw = math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))
    return np.array([roll, pitch, yaw], float)


def euler_to_quat(rpy) -> np.ndarray:
    r, p, y = [float(v) for v in np.asarray(rpy, float).ravel()[:3]]
    cr, sr = math.cos(r / 2), math.sin(r / 2)
    cp, sp = math.cos(p / 2), math.sin(p / 2)
    cy, sy = math.cos(y / 2), math.sin(y / 2)
    return np.array([cr * cp * cy + sr * sp * sy, sr * cp * cy - cr * sp * sy,
                     cr * sp * cy + sr * cp * sy, cr * cp * sy - sr * sp * cy], float)


def geodesic_angle_deg(q_a, q_b) -> float:
    """Shortest rotation angle between two wxyz quaternions [deg]."""
    a = np.asarray(q_a, float).ravel()[:4]
    b = np.asarray(q_b, float).ravel()[:4]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return float("nan")
    dot = float(np.clip(abs(float(np.dot(a / na, b / nb))), -1.0, 1.0))
    return math.degrees(2.0 * math.acos(dot))


def _deriv(x, t):
    return np.gradient(np.asarray(x, float), np.asarray(t, float), axis=0)

### Per-episode metric extraction

In [5]:
@dataclass
class Episode:
    path: str
    stem: str
    task: str = "unknown"
    instruction: str = ""
    success: Any = None
    stop_reason: str = "?"

    n_ticks: int = 0
    n_vla_steps: int = 0
    dt: Any = None
    duration_sim_s: float = float("nan")
    duration_wall_s: float = float("nan")

    ik_counts: dict = field(default_factory=dict)
    ik_pct: dict = field(default_factory=dict)
    ik_success_pct: float = float("nan")
    ik_failure_pct: float = float("nan")
    raw_stage_counts: dict = field(default_factory=dict)

    ik_pos_err_mm: Any = None
    ik_rot_err_deg: Any = None
    lat_vla_ms: Any = None
    lat_ik_ms: Any = None
    lat_exec_ms: Any = None
    track_pos_err_mm: Any = None
    track_rot_err_deg: Any = None
    vel: Any = None
    acc: Any = None
    jerk: Any = None

    hard_collision_ticks: int = 0
    warn_collision_ticks: int = 0
    min_clearance_m: float = float("nan")
    has_collision_data: bool = False


def load_episode(path) -> Episode:
    """Read one .h5 log and derive every metric needed by Groups B and C."""
    path = Path(path)
    ep = Episode(path=str(path), stem=path.stem)

    with h5py.File(path, "r") as f:
        ep.task = _txt(_attr(f, "preset_name", "task_name", default=path.parent.name))
        ep.instruction = _txt(_attr(f, "instruction", default="")) or ep.task.replace("_", " ")
        ep.stop_reason = _txt(_attr(f, "stop_reason", default="?"))

        succ = _attr(f, "success", default=None)
        if succ is None and "success" in f:
            succ = np.atleast_1d(f["success"][()])[-1]
        if succ is not None:
            if isinstance(succ, (str, bytes)):
                ep.success = _txt(succ).strip().lower() in ("1", "true", "success", "yes")
            else:
                ep.success = bool(np.atleast_1d(succ)[-1])

        # time base: uniform physics grid, wall clock only as fallback  
        dt = _attr(f, "physics_dt", default=None)
        ep.dt = float(dt) if dt is not None else None
        q_act = _get(f, "ticks/q_actual", "q_actual")
        eef_pos = _get(f, "ticks/eef_pos", "eef_pos")
        ref = next((a for a in (q_act, eef_pos) if a is not None), None)
        ep.n_ticks = int(ref.shape[0]) if ref is not None else 0
        t_wall = _get(f, "ticks/t_wall")
        if ep.dt and ep.n_ticks:
            t = np.arange(ep.n_ticks) * ep.dt
        elif t_wall is not None and len(t_wall):
            t = np.asarray(t_wall, float) - float(t_wall[0])
        else:
            t = np.arange(ep.n_ticks, dtype=float)
        ep.duration_sim_s = float(t[-1]) if len(t) else float("nan")
        wall = _attr(f, "completion_time_wall_s", default=None)
        ep.duration_wall_s = float(wall) if wall is not None else float("nan")

        # VLA step count  
        acts = _get(f, "vla_steps/exec_action", "vla_steps/raw_action")
        vla_t = _get(f, "vla_steps/t_wall")
        ep.n_vla_steps = int(len(acts)) if acts is not None else (int(len(vla_t)) if vla_t is not None else 0)

        # IK hierarchy  
        stages = _get(f, "vla_steps/ik_stage", "vla_steps/ik_status", "vla_steps/escalation_reason")
        if stages is None and "ik_stage_counts_json" in f.attrs:
            raw = json.loads(_txt(f.attrs["ik_stage_counts_json"]))
            stages = np.array([k for k, c in raw.items() for _ in range(int(c))], dtype=object)
        if stages is not None and len(stages):
            names = [_txt(s) for s in stages]
            ep.raw_stage_counts = {k: int(v) for k, v in Counter(names).items()}
            cats = [classify_ik_stage(s) for s in names]
            counts = Counter(cats)
            total = max(len(cats), 1)
            ep.ik_counts = {k: int(counts.get(k, 0)) for k in IK_ALL_KEYS}
            ep.ik_pct = {k: 100.0 * ep.ik_counts[k] / total for k in IK_ALL_KEYS}
            ep.ik_success_pct = sum(ep.ik_pct[k] for k in IK_SUCCESS_KEYS)
            ep.ik_failure_pct = sum(ep.ik_pct[k] for k in IK_FAILURE_KEYS)

        # IK accuracy (solver residuals)  
        pos_err = _get(f, "vla_steps/ik_pos_err", "ik_pos_err")
        rot_err = _get(f, "vla_steps/ik_rot_err", "ik_rot_err")
        if pos_err is not None:
            ep.ik_pos_err_mm = np.asarray(pos_err, float).ravel() * 1000.0
        if rot_err is not None:
            ep.ik_rot_err_deg = np.degrees(np.asarray(rot_err, float).ravel())

        # latency  
        vla_ms = _get(f, "vla_steps/vla_inference_ms", "vla_steps/infer_ms")
        ik_ms = _get(f, "vla_steps/ik_solve_ms", "ik_solve_ms")
        exec_ms = _get(f, "vla_steps/step_wall_ms", "vla_steps/exec_ms",
                       "vla_steps/step_ms", "vla_steps/total_ms", "vla_steps/loop_ms")
        if vla_ms is not None:
            ep.lat_vla_ms = np.asarray(vla_ms, float).ravel()
        if ik_ms is not None:
            ep.lat_ik_ms = np.asarray(ik_ms, float).ravel()
        if exec_ms is not None:
            ep.lat_exec_ms = np.asarray(exec_ms, float).ravel()
        elif vla_t is not None and len(vla_t) > 1:      # wall-clock step period as fallback
            ep.lat_exec_ms = np.diff(np.asarray(vla_t, float).ravel()) * 1000.0

        # Cartesian tracking (commanded target vs achieved)  
        pc = _get(f, "vla_steps/commanded_target_pos")
        pa = _get(f, "vla_steps/eef_pos_after")
        qc = _get(f, "vla_steps/commanded_target_quat")
        ra = _get(f, "vla_steps/eef_rpy_after")
        te_p = _get(f, "vla_steps/tracking_pos_err")
        te_r = _get(f, "vla_steps/tracking_rot_err")
        if pc is not None and pa is not None:
            n = min(len(pc), len(pa))
            ep.track_pos_err_mm = np.linalg.norm(np.asarray(pc[:n], float) - np.asarray(pa[:n], float),
                                                 axis=-1) * 1000.0
        elif te_p is not None:
            ep.track_pos_err_mm = np.asarray(te_p, float).ravel() * 1000.0
        if qc is not None and np.ndim(qc) == 2 and np.shape(qc)[-1] == 4 and ra is not None:
            n = min(len(qc), len(ra))
            ep.track_rot_err_deg = np.array(
                [geodesic_angle_deg(qc[i], euler_to_quat(ra[i])) for i in range(n)], float)
        elif te_r is not None:
            ep.track_rot_err_deg = np.degrees(np.asarray(te_r, float).ravel())

        # smoothness (Cartesian velocity / acceleration / jerk magnitude)  
        if eef_pos is not None and len(eef_pos) >= 4 and len(t) >= 4:
            p = np.asarray(eef_pos, float)
            tt = t[:len(p)]
            v = _deriv(p, tt)
            a = _deriv(v, tt)
            j = _deriv(a, tt)
            keep = tt >= SMOOTHNESS_START_S
            if not keep.any():
                keep = np.ones(len(tt), bool)
            ep.vel = np.linalg.norm(v, axis=-1)[keep]
            ep.acc = np.linalg.norm(a, axis=-1)[keep]
            ep.jerk = np.linalg.norm(j, axis=-1)[keep]

        # safety / collisions  
        hard = _get(f, "ticks/collision_hard", "collision_hard")
        mind = _get(f, "ticks/collision_min_dist", "collision_min_dist")
        warn = _get(f, "ticks/collision_warn", "collision_warn")
        if hard is not None:
            ep.has_collision_data = True
            ep.hard_collision_ticks = int(np.asarray(hard).astype(bool).sum())
        if mind is not None:
            d = np.asarray(mind, float).ravel()
            d = d[np.isfinite(d)]
            if d.size:
                ep.has_collision_data = True
                ep.min_clearance_m = float(d.min())
                ep.warn_collision_ticks = int((d < WARN_CLEARANCE_M).sum())
        if warn is not None:
            ep.has_collision_data = True
            ep.warn_collision_ticks = max(ep.warn_collision_ticks,
                                          int(np.asarray(warn).astype(bool).sum()))
    return ep


def load_trajectories(path) -> dict:
    """Heavy per-tick / per-step arrays, loaded only for the representative episode."""
    out: dict = {}
    with h5py.File(path, "r") as f:
        dt = _attr(f, "physics_dt", default=None)
        q_act = _get(f, "ticks/q_actual", "q_actual")
        q_cmd = _get(f, "ticks/q_cmd", "ticks/q_des", "q_cmd")
        eef = _get(f, "ticks/eef_pos", "eef_pos")
        n = len(q_act) if q_act is not None else (len(eef) if eef is not None else 0)
        t_wall = _get(f, "ticks/t_wall")
        if dt and n:
            t = np.arange(n) * float(dt)
        elif t_wall is not None and len(t_wall):
            t = np.asarray(t_wall, float) - float(t_wall[0])
        else:
            t = np.arange(n, dtype=float)
        out.update(t=t, q_actual=q_act, q_cmd=q_cmd, eef_pos=eef,
                   eef_pos_after=_get(f, "vla_steps/eef_pos_after"),
                   commanded_target_pos=_get(f, "vla_steps/commanded_target_pos"))
    return out

### Preset aggregation

In [6]:
@dataclass
class Preset:
    name: str
    directory: str
    instruction: str
    episodes: list

    success_rate: float = float("nan")
    success_ci: float = float("nan")
    n_success: int = 0
    n_scored: int = 0
    steps: Stat = field(default_factory=Stat)
    time: Stat = field(default_factory=Stat)
    ik_success: Stat = field(default_factory=Stat)
    ik_stage: dict = field(default_factory=dict)
    ik_pos_err: Stat = field(default_factory=Stat)
    ik_rot_err: Stat = field(default_factory=Stat)
    lat: dict = field(default_factory=dict)
    track_pos: Stat = field(default_factory=Stat)
    track_rot: Stat = field(default_factory=Stat)
    smooth: dict = field(default_factory=dict)
    representative: Any = None

    @property
    def n_episodes(self) -> int:
        return len(self.episodes)

    @property
    def pretty(self) -> str:
        return self.name.replace("_", " ")


def build_preset(directory: Path, paths: Sequence[Path], verbose: bool = True) -> Preset:
    eps = []
    for p in paths:
        try:
            eps.append(load_episode(p))
        except Exception as exc:                                   # keep the batch alive
            print(f"  [WARN] skipping {Path(p).name}: {exc}")
    if not eps:
        raise RuntimeError(f"no readable .h5 episodes in {directory}")

    name = Counter(e.task for e in eps).most_common(1)[0][0] or directory.name
    instruction = next((e.instruction for e in eps if e.instruction), name.replace("_", " "))
    ps = Preset(name=name, directory=str(directory), instruction=instruction, episodes=eps)

    # task success
    flags = [e.success for e in eps if e.success is not None]
    ps.n_scored = len(flags)
    if flags:
        v = np.array([1.0 if x else 0.0 for x in flags], float)
        ps.success_rate = 100.0 * float(v.mean())
        ps.n_success = int(v.sum())
        if v.size > 1:
            ps.success_ci = 100.0 * ci95_factor(v.size) * v.std(ddof=1) / math.sqrt(v.size)
    ok = [e for e in eps if e.success] or eps          # completion stats over successes
    ps.steps = episode_stat([e.n_vla_steps for e in ok])
    ps.time = episode_stat([e.duration_wall_s if np.isfinite(e.duration_wall_s) else e.duration_sim_s
                            for e in ok])

    # IK performance
    ps.ik_success = episode_stat([e.ik_success_pct for e in eps])
    ps.ik_stage = {k: episode_stat([e.ik_pct.get(k, np.nan) for e in eps]) for k in IK_ALL_KEYS}

    # IK accuracy
    ps.ik_pos_err = pooled_stat([e.ik_pos_err_mm for e in eps])
    ps.ik_rot_err = pooled_stat([e.ik_rot_err_deg for e in eps])

    # latency 
    ps.lat = {"vla": pooled_stat([e.lat_vla_ms for e in eps]),
              "ik": pooled_stat([e.lat_ik_ms for e in eps]),
              "exec": pooled_stat([e.lat_exec_ms for e in eps])}

    # tracking
    ps.track_pos = pooled_stat([e.track_pos_err_mm for e in eps])
    ps.track_rot = pooled_stat([e.track_rot_err_deg for e in eps])

    # smoothness
    ps.smooth = {"vel": pooled_stat([e.vel for e in eps]),
                 "acc": pooled_stat([e.acc for e in eps]),
                 "jerk": pooled_stat([e.jerk for e in eps])}

    ps.representative = pick_representative(eps)
    if verbose:
        rep = ps.representative.stem if ps.representative else "n/a"
        print(f"  preset '{ps.name}': {ps.n_episodes} episodes, success "
              f"{ps.success_rate:.0f}% ({ps.n_success}/{ps.n_scored}), representative = {rep}")
    return ps


def pick_representative(eps: Sequence[Episode]):
    """Successful episode closest to the batch median tracking error (a medoid, not a
    cherry-picked best run). Falls back to median duration, then the first episode."""
    pool = [e for e in eps if e.success] or list(eps)

    def score(e: Episode) -> float:
        for arr in (e.track_pos_err_mm, e.ik_pos_err_mm):
            if arr is not None and np.isfinite(arr).any():
                return float(np.nanmean(arr))
        return float("nan")

    valid = [(score(e), e) for e in pool]
    valid = [(s, e) for s, e in valid if np.isfinite(s)]
    if valid:
        med = float(np.median([s for s, _ in valid]))
        return min(valid, key=lambda se: abs(se[0] - med))[1]
    durs = [(e.duration_sim_s, e) for e in pool if np.isfinite(e.duration_sim_s)]
    if durs:
        med = float(np.median([d for d, _ in durs]))
        return min(durs, key=lambda de: abs(de[0] - med))[1]
    return pool[0] if pool else None

### Figure scaffolding - unified thesis style

In [7]:
class Exporter:
    def __init__(self, out_root: Path, dpi: int = 320, formats: Sequence[str] = ("png", "pdf")):
        self.root = Path(out_root)
        self.fig_dir = self.root / "figures"
        self.tab_dir = self.root / "tables"
        self.dpi = dpi
        self.formats = tuple(formats)
        for d in (self.fig_dir, self.tab_dir):
            d.mkdir(parents=True, exist_ok=True)
        self.written: list = []

    def figure(self, fig, name: str, subdir=None) -> Path:
        base = self.fig_dir / subdir if subdir else self.fig_dir
        base.mkdir(parents=True, exist_ok=True)
        for ext in self.formats:
            fig.savefig(base / f"{name}.{ext}", dpi=self.dpi, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        out = base / f"{name}.{self.formats[0]}"
        self.written.append(str(out))
        print(f"  [fig]   {base / name}.{'|'.join(self.formats)}")
        return out

    def table(self, df: pd.DataFrame, name: str, subdir=None, caption=None) -> Path:
        base = self.tab_dir / subdir if subdir else self.tab_dir
        base.mkdir(parents=True, exist_ok=True)
        csv = base / f"{name}.csv"
        df.to_csv(csv, index=False)
        try:
            md = df.to_markdown(index=False)
        except Exception:
            md = df.to_string(index=False)
        (base / f"{name}.md").write_text((f"**{caption}**\n\n" if caption else "") + md + "\n",
                                         encoding="utf-8")
        try:
            (base / f"{name}.tex").write_text(
                df.to_latex(index=False, escape=True, caption=caption, label=f"tab:{name}"),
                encoding="utf-8")
        except Exception:
            pass
        self.written.append(str(csv))
        print(f"  [table] {base / name}.csv|.md|.tex")
        return csv


def two_line_title(fig, main: str, instruction: str = "", *, top: float = 0.94,
                   main_size: float = 14.0, sub_size: float = 10.0, y: float = 0.985,
                   layout: bool = True) -> None:
    """Single bold scientific title, tightly hugging the axes below it.

    `instruction` is accepted for call-site compatibility but is intentionally not
    rendered (thesis figures show only the scientific title - no second subtitle line).
    """
    fig.suptitle(main, fontsize=main_size, fontweight="bold", y=y, va="top")
    if layout:
        fig.tight_layout(rect=(0, 0, 1, top))


def style_ax(ax, *, xlabel=None, ylabel=None, title=None, legend=False, yticks5=True):
    ax.grid(True, color=".8", linewidth=0.7)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=10)
    if title:
        ax.set_title(title, fontsize=11, fontweight="semibold", pad=6)
    ax.tick_params(labelsize=8.5)
    if yticks5:
        try:
            ax.yaxis.set_major_locator(MaxNLocator(5))
        except Exception:
            pass
    if legend:
        h, _ = ax.get_legend_handles_labels()
        if h:
            ax.legend(fontsize=9, loc="best")
    for sp in ax.spines.values():
        sp.set_color("0.15")


def _decimate(a, cap: int = MAX_VIOLIN_SAMPLES) -> np.ndarray:
    a = np.asarray(a, float).ravel()
    a = a[np.isfinite(a)]
    if a.size <= cap:
        return a
    return a[:: max(1, a.size // cap)]


def _violin(ax, data, positions, *, colors, horizontal=True, widths=0.72):
    kw = dict(positions=positions, widths=widths, showextrema=False, showmedians=False)
    series = [_decimate(d) for d in data]
    try:                                    # matplotlib >= 3.10
        parts = ax.violinplot(series, orientation="horizontal" if horizontal else "vertical", **kw)
    except TypeError:                       # older matplotlib
        parts = ax.violinplot(series, vert=not horizontal, **kw)
    for body, c in zip(parts["bodies"], colors):
        body.set_facecolor(c)
        body.set_edgecolor("0.2")
        body.set_alpha(0.60)          # translucent: overlaid stat markers stay readable
        body.set_linewidth(0.9)
    return parts

## 2. Figures & Tables & Input Driver Resolver

### quantitative figures

In [8]:
def _pct_row(pooled_pct: float, st: Stat) -> dict:
    return {"Share of calls (%)": round(pooled_pct, 2),
            "Mean +/- Std across episodes (%)": st.pm_std(2),
            "95% CI (%)": st.ci_str(2)}


def fig_1_ik_donut(ps: Preset, ex: Exporter):
    """Nested donut: outer = successful/failed IK, inner = the four hierarchy levels."""
    counts = {k: sum(e.ik_counts.get(k, 0) for e in ps.episodes) for k in IK_ALL_KEYS}
    total = sum(counts.values())
    if total == 0:
        print("  [skip]  B1 - no IK stage data")
        return None

    inner_keys = [k for k in IK_ALL_KEYS if counts[k] > 0]
    succ = sum(counts[k] for k in IK_SUCCESS_KEYS)
    fail = total - succ
    outer_vals, outer_lab, outer_col = [], [], []
    if succ:
        outer_vals.append(succ); outer_lab.append("Successful IK"); outer_col.append(IK_COLORS["success"])
    if fail:
        outer_vals.append(fail); outer_lab.append("Failed IK"); outer_col.append(IK_COLORS["failure"])

    fig, ax = plt.subplots(figsize=FIG_DONUT)
    ax.set_axis_off()
    _, outer_label_texts, outer_pct_texts = ax.pie(
        outer_vals, radius=1.0, colors=outer_col, startangle=90, counterclock=False,
        labels=outer_lab,
        autopct=lambda p: f"{p:.1f}%",
        pctdistance=0.85,              # percentage centered inside the outer ring
        labeldistance=1.08,            # Successful IK / Failed IK stay outside
        wedgeprops=dict(width=0.30, edgecolor="white", linewidth=1.6),
    )
    for txt in outer_label_texts:
        txt.set_fontsize(8.0)
        txt.set_fontweight("semibold")
        txt.set_color("0.15")
    for txt in outer_pct_texts:
        txt.set_fontsize(7.4)
        txt.set_fontweight("bold")
        txt.set_color("white")

    ax.pie([counts[k] for k in inner_keys], radius=0.70,
           colors=[IK_COLORS[k] for k in inner_keys], startangle=90, counterclock=False,
           autopct=lambda p: f"{p:.1f}%" if p >= 4 else "", pctdistance=0.76,
           textprops=dict(fontsize=7.2, color="white", fontweight="bold"),
           wedgeprops=dict(width=0.32, edgecolor="white", linewidth=1.4))
    legend_handles = [Patch(facecolor=IK_COLORS[k], edgecolor="white",
                        label=IK_LABELS[k]) for k in inner_keys]
    ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.04),
              ncol=max(1, len(inner_keys)), fontsize=7.8, frameon=True, title="IK solution strategy",
              title_fontsize=8.0, handletextpad=0.55, borderaxespad=0.0)
    ax.set_aspect("equal")
    two_line_title(fig, "IK solution strategy distribution", top=0.95,
                   main_size=12.5, y=0.96, layout=False)
    fig.subplots_adjust(left=0.08, right=0.92, bottom=0.20, top=0.91)
    ex.figure(fig, "1_ik_solution_strategy_distribution", subdir=ps.name)

    rows = [dict(Category="Successful IK", Level="all", Calls=succ,
                 **_pct_row(100.0 * succ / total, ps.ik_success)),
            dict(Category="Failed IK", Level="all", Calls=fail,
                 **_pct_row(100.0 * fail / total,
                            episode_stat([e.ik_failure_pct for e in ps.episodes])))]
    for k in inner_keys:
        rows.append(dict(Category="Successful" if k in IK_SUCCESS_KEYS else "Failed",
                         Level=IK_LABELS[k], Calls=counts[k],
                         **_pct_row(100.0 * counts[k] / total, ps.ik_stage[k])))
    df = pd.DataFrame(rows)
    ex.table(df, "1_ik_solution_strategy_stats", subdir=ps.name,
             caption=f"IK solution strategy distribution - {ps.pretty} (n={ps.n_episodes} episodes)")
    return df


def fig_2_latency(ps: Preset, ex: Exporter):
    """Horizontal violins: VLA inference / IK solving / complete execution latency."""
    spec = [("vla", "VLA inference", C_BLUE, [e.lat_vla_ms for e in ps.episodes]),
            ("ik", "IK solving", C_GREEN, [e.lat_ik_ms for e in ps.episodes]),
            ("exec", "Complete execution", C_PURPLE, [e.lat_exec_ms for e in ps.episodes])]
    avail = [(k, lab, col, np.concatenate([np.asarray(a, float).ravel()
                                           for a in arrs if a is not None and len(a)]))
             for k, lab, col, arrs in spec if any(a is not None and len(a) for a in arrs)]
    if not avail:
        print("  [skip]  2 - no latency data")
        return None

    fig, ax = plt.subplots(figsize=FIG_VIOLIN_H)
    pos = np.arange(len(avail))
    _violin(ax, [d for *_, d in avail], pos, colors=[c for _, _, c, _ in avail], horizontal=True)
    xmax = max(float(np.nanmax(d)) for *_, d in avail)
    ax.set_xlim(0, xmax * 1.04)
    for y, (k, lab, col, d) in zip(pos, avail):
        st = ps.lat[k]
        ax.plot([st.p5, st.p95], [y, y], color="0.15", lw=1.6, zorder=3)
        ax.plot([st.p5, st.p95], [y, y], "|", color="0.15", ms=9, mew=1.6, zorder=3)
        ax.plot(st.median, y, "o", color="white", mec="0.1", ms=7.5, mew=1.4, zorder=4)
    ax.set_yticks(pos)
    ax.set_yticklabels([lab for _, lab, _, _ in avail], fontsize=10)
    ax.set_ylim(len(avail) - 0.4, -0.7)
    style_ax(ax, xlabel="Latency [ms]", yticks5=False)
    ax.grid(axis="y", visible=False)
    median_handles = [plt.Line2D([], [], marker="o", ls="none", color=col, mec="0.1",
                                 mew=1.0, ms=7,
                                 label=f"median {ps.lat[k].median:.1f} ms")
                      for k, lab, col, _ in avail]
    two_line_title(fig, "Computational latency distribution", top=0.90)
    ax.legend(handles=median_handles, fontsize=8.3, loc="upper right",
              handletextpad=0.5, borderaxespad=0.7)
    fig.subplots_adjust(top=0.87, bottom=0.13)
    ex.figure(fig, "2_computational_latency_distribution", subdir=ps.name)

    df = pd.DataFrame([{"Stage": lab, "Samples": ps.lat[k].n_samples,
                        "Mean +/- Std (ms)": ps.lat[k].pm_std(2),
                        "Median (ms)": round(ps.lat[k].median, 2),
                        "P5 (ms)": round(ps.lat[k].p5, 2),
                        "P95 (ms)": round(ps.lat[k].p95, 2),
                        "95% CI of mean (ms)": ps.lat[k].ci_str(2)}
                       for k, lab, _, _ in avail])
    ex.table(df, "2_latency_stats", subdir=ps.name,
             caption=f"Computational latency - {ps.pretty} (n={ps.n_episodes} episodes)")
    return df


def _error_histogram(ps: Preset, ex: Exporter, *, arrays, st: Stat, xlabel: str, main: str,
                     color: str, fname: str, unit: str, prec: int = 2):
    data = [np.asarray(a, float).ravel() for a in arrays if a is not None and len(a)]
    if not data:
        print(f"  [skip]  {fname} - no data")
        return None
    pooled = np.concatenate(data)
    pooled = pooled[np.isfinite(pooled)]
    if not pooled.size:
        print(f"  [skip]  {fname} - no finite samples")
        return None

    bins = int(np.clip(np.sqrt(pooled.size), 20, 60))
    fig, ax = plt.subplots(figsize=FIG_HIST)
    ax.hist(pooled, bins=bins, color=color, edgecolor="white", linewidth=0.6, alpha=0.92)
    ax.axvline(st.mean, color="#C92A2A", lw=1.6, label=f"mean = {st.mean:.{prec}f} {unit}")
    ax.axvline(st.median, color="0.25", lw=1.3, ls="--",
               label=f"median = {st.median:.{prec}f} {unit}")
    if np.isfinite(st.lo) and np.isfinite(st.hi):
        ax.axvspan(st.lo, st.hi, color="#C92A2A", alpha=0.13, lw=0,
                   label=f"95% CI = {st.ci_str(prec)} {unit}")
    style_ax(ax, xlabel=xlabel, ylabel="Frequency [count]")
    ax.legend(fontsize=8.5, loc="upper right")
    span = float(np.ptp(pooled)) or 1.0
    ax.set_xlim(left=max(0.0, float(pooled.min()) - 0.02 * span))
    two_line_title(fig, main, top=0.90)
    fig.subplots_adjust(top=0.88)
    ex.figure(fig, fname, subdir=ps.name)

    df = pd.DataFrame([{"Metric": xlabel, "Samples": st.n_samples, "Episodes": st.n_episodes,
                        f"Mean ({unit})": round(st.mean, prec + 1),
                        f"Std ({unit})": round(st.std, prec + 1),
                        f"Median ({unit})": round(st.median, prec + 1),
                        f"P95 ({unit})": round(st.p95, prec + 1),
                        f"Max ({unit})": round(float(pooled.max()), prec + 1),
                        "95% CI of mean": st.ci_str(prec + 1)}])
    ex.table(df, f"{fname}_stats", subdir=ps.name,
             caption=f"{main} - {ps.pretty} (n={ps.n_episodes} episodes)")
    return df


def fig_3_pos_error(ps: Preset, ex: Exporter):
    return _error_histogram(ps, ex, arrays=[e.ik_pos_err_mm for e in ps.episodes],
                            st=ps.ik_pos_err, xlabel="Position error [mm]",
                            main="IK position error distribution", color=C_BLUE,
                            fname="3_ik_position_error_distribution", unit="mm", prec=2)


def fig_4_rot_error(ps: Preset, ex: Exporter):
    return _error_histogram(ps, ex, arrays=[e.ik_rot_err_deg for e in ps.episodes],
                            st=ps.ik_rot_err, xlabel="Orientation error [degrees]",
                            main="IK orientation error distribution", color=C_CYAN,
                            fname="4_ik_orientation_error_distribution", unit="deg", prec=2)


def _equal_aspect_3d(ax, pts: np.ndarray) -> None:
    lo, hi = pts.min(axis=0), pts.max(axis=0)
    ctr = (lo + hi) / 2.0
    r = max(float(np.max(hi - lo)) / 2.0, 1e-3) * 1.15
    ax.set_xlim(ctr[0] - r, ctr[0] + r)
    ax.set_ylim(ctr[1] - r, ctr[1] + r)
    ax.set_zlim(ctr[2] - r, ctr[2] + r)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass


def fig_5_eef_tracking(ps: Preset, ex: Exporter):
    """3D desired vs executed EEF trajectory, one representative successful episode."""
    rep = ps.representative
    if rep is None:
        print("  [skip]  5 - no representative episode")
        return None
    tr = load_trajectories(rep.path)
    des = tr.get("commanded_target_pos")
    exe = tr.get("eef_pos_after")
    tick = tr.get("eef_pos")
    if exe is None and tick is not None:
        exe = tick
    if exe is None:
        print("  [skip]  5 - no executed EEF trajectory")
        return None
    exe = np.asarray(exe, float)
    des = np.asarray(des, float) if des is not None else None

    fig = plt.figure(figsize=(8.0, 6.6))
    ax = fig.add_subplot(111, projection="3d")
    if tick is not None and len(tick) and not np.shares_memory(np.asarray(tick), exe):
        tk = np.asarray(tick, float)
        if tk.shape != exe.shape or not np.allclose(tk, exe):
            ax.plot(tk[:, 0], tk[:, 1], tk[:, 2], color="0.72", lw=0.9, alpha=0.85,
                    label="Executed (tick resolution)")
    if des is not None and des.ndim == 2:
        n = min(len(des), len(exe))
        ax.plot(des[:n, 0], des[:n, 1], des[:n, 2], color=C_RED, lw=1.7, ls="--",
                label="Desired EEF trajectory")
    ax.plot(exe[:, 0], exe[:, 1], exe[:, 2], color=C_BLUE, lw=1.9, label="Executed EEF trajectory")
    ax.scatter(*exe[0], color=C_GREEN, s=55, edgecolors="white", lw=0.8, depthshade=False,
               label="Start", zorder=5)
    ax.scatter(*exe[-1], color="#C92A2A", s=55, marker="X", edgecolors="white", lw=0.8,
               depthshade=False, label="End", zorder=5)
    ax.set_xlabel("x [m]", fontsize=10, labelpad=6)
    ax.set_ylabel("y [m]", fontsize=10, labelpad=6)
    ax.set_zlabel("z [m]", fontsize=10, labelpad=2)
    ax.tick_params(labelsize=8)
    _equal_aspect_3d(ax, np.vstack([exe] + ([des] if des is not None and des.ndim == 2 else [])))
    ax.view_init(elev=22, azim=-58)
    ax.legend(fontsize=8.5, loc="upper left", bbox_to_anchor=(0.0, 1.0), borderaxespad=0.2,
              framealpha=0.92)
    two_line_title(fig, "End-effector trajectory tracking performance", top=0.94, layout=False)
    fig.subplots_adjust(left=0.02, right=0.90, bottom=0.03, top=0.94)
    ex.figure(fig, "5_eef_trajectory_tracking", subdir=ps.name)

    per_ep_max = [float(np.nanmax(e.track_pos_err_mm)) for e in ps.episodes
                  if e.track_pos_err_mm is not None and np.isfinite(e.track_pos_err_mm).any()]
    df = pd.DataFrame([{"Task preset": ps.pretty, "Episodes": ps.n_episodes,
                        "Mean tracking error (mm)": round(ps.track_pos.mean, 3),
                        "Std (mm)": round(ps.track_pos.std, 3),
                        "Max error (mm)": round(max(per_ep_max), 3) if per_ep_max else float("nan"),
                        "95% CI of mean (mm)": ps.track_pos.ci_str(3),
                        "Mean orientation error (deg)": round(ps.track_rot.mean, 3),
                        "Std (deg)": round(ps.track_rot.std, 3),
                        "Success rate (%)": round(ps.success_rate, 1),
                        "Representative episode": rep.stem}])
    ex.table(df, "5_eef_tracking_summary", subdir=ps.name,
             caption=f"End-effector tracking batch summary - {ps.pretty} (n={ps.n_episodes} episodes)")
    return df


def fig_6_joint_tracking(ps: Preset, ex: Exporter):
    """Seven joints in a 4 + 3 grid (no empty eighth axis), commanded vs actual."""
    rep = ps.representative
    if rep is None:
        return None
    tr = load_trajectories(rep.path)
    q = tr.get("q_actual")
    qc = tr.get("q_cmd")
    if q is None:
        print("  [skip]  6 - no joint positions")
        return None
    q = np.asarray(q, float)
    t = np.asarray(tr["t"], float)[:len(q)]
    qc = np.asarray(qc, float) if qc is not None else None
    n_j = min(7, q.shape[1])

    fig = plt.figure(figsize=FIG_JOINTS)
    # 24-column grid: top row = four equal-width (6-col) slots; bottom row's three slots
    # are also 6 columns wide but shifted by half a slot, so each one sits exactly centred
    # between the two top-row subplots above it.
    gs = fig.add_gridspec(2, 24, hspace=0.55, wspace=2.6)
    slots = [gs[0, 0:6], gs[0, 6:12], gs[0, 12:18], gs[0, 18:24],   # row 1: joints 1-4
             gs[1, 3:9], gs[1, 9:15], gs[1, 15:21]]                 # row 2: joints 5-7 (centred)
    handles, labels, rows = None, None, []
    for i in range(n_j):
        ax = fig.add_subplot(slots[i])
        ax.plot(t, q[:, i], color=C_BLUE, lw=1.7, label="Actual", zorder=2)
        if qc is not None and len(qc) >= len(q) and qc.shape[1] > i:
            ax.plot(t, qc[:len(q), i], color=C_RED, lw=1.0, ls=(0, (4, 3)), label="Commanded",
                    alpha=0.95, zorder=3)
            err = np.abs(qc[:len(q), i] - q[:, i])
            rows.append({"Joint": JOINT_NAMES[i],
                         "Mean |error| (rad)": round(float(err.mean()), 5),
                         "Std (rad)": round(float(err.std(ddof=1)), 5),
                         "Max |error| (rad)": round(float(err.max()), 5),
                         "RMS (rad)": round(float(np.sqrt((err ** 2).mean())), 5),
                         "Mean |error| (deg)": round(float(np.degrees(err.mean())), 4)})
        style_ax(ax, xlabel="Time [s]", title=JOINT_NAMES[i])
        if i in (0, 4):                                  # y-label once per row
            ax.set_ylabel("Joint angle [rad]", fontsize=10)
        if handles is None:
            handles, labels = ax.get_legend_handles_labels()
    two_line_title(fig, "Joint trajectory tracking performance", top=0.92, layout=False)
    fig.subplots_adjust(left=0.045, right=0.985, top=0.89, bottom=0.155)
    if handles:
        fig.legend(handles, labels, loc="lower center", ncol=2, fontsize=10,
                   bbox_to_anchor=(0.5, 0.0), frameon=True)
    ex.figure(fig, "6_joint_trajectory_tracking", subdir=ps.name)

    if rows:
        df = pd.DataFrame(rows)
        ex.table(df, "6_joint_tracking_error", subdir=ps.name,
                 caption=f"Joint tracking error, representative episode {rep.stem} - {ps.pretty}")
        return df
    return None


def fig_7_smoothness(ps: Preset, ex: Exporter):
    """Readable histograms of Cartesian velocity, acceleration and jerk magnitude."""
    spec = [("vel", "Velocity magnitude", "|v| [m/s]", C_BLUE, [e.vel for e in ps.episodes]),
            ("acc", "Acceleration magnitude", "|a| [m/s$^2$]", C_RED, [e.acc for e in ps.episodes]),
            ("jerk", "Jerk magnitude", "|j| [m/s$^3$]", C_GREEN, [e.jerk for e in ps.episodes])]
    units = {"vel": "m/s", "acc": "m/s^2", "jerk": "m/s^3"}
    avail = [(k, lab, unit, col, np.concatenate([np.asarray(a, float).ravel()
                                                 for a in arrs if a is not None and len(a)]))
             for k, lab, unit, col, arrs in spec if any(a is not None and len(a) for a in arrs)]
    if not avail:
        print("  [skip]  7 - no smoothness data")
        return None

    fig, axs = plt.subplots(1, len(avail), figsize=FIG_SMOOTHNESS, sharey=True)
    axs = np.atleast_1d(axs)
    for ax, (k, lab, unit, col, d) in zip(axs, avail):
        d = d[np.isfinite(d)]
        st = ps.smooth[k]
        xcap = float(np.percentile(d, 99.5))
        if not np.isfinite(xcap) or xcap <= 0:
            xcap = float(np.max(d)) if d.size and np.max(d) > 0 else 1.0
        shown = d[d <= xcap]
        bins = np.linspace(0.0, xcap, 31)
        # Percent of all samples per bin is easier to interpret than KDE width.
        weights = np.full(shown.size, 100.0 / d.size)
        ax.hist(shown, bins=bins, weights=weights, color=col, alpha=0.78,
                edgecolor="white", linewidth=0.45)
        ax.axvspan(st.p5, st.p95, color=col, alpha=0.10, zorder=0)
        ax.axvline(st.p5, color="0.35", ls=":", lw=1.0, zorder=3)
        ax.axvline(st.p95, color="0.35", ls=":", lw=1.0, zorder=3)
        ax.axvline(st.median, color="0.08", lw=1.8, zorder=4)
        ax.set_xlim(0, xcap)
        style_ax(ax, xlabel=unit, title=lab, yticks5=True)
        if k == "vel":
            # Display velocity ticks in units of 10^-2 m/s (for example, 0.05 -> 5).
            ax.ticklabel_format(axis="x", style="sci", scilimits=(-2, -2), useMathText=True)
            ax.xaxis.get_offset_text().set_fontsize(9)
        ax.grid(axis="x", visible=False)
        ax.text(0.97, 0.96,
                f"median {st.median:.3g}\n90% range {st.p5:.3g}-{st.p95:.3g}",
                transform=ax.transAxes, ha="right", va="top", fontsize=8.2,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc", alpha=0.94))
    axs[0].set_ylabel("Samples per bin [%]")
    two_line_title(fig, "Motion smoothness distributions", top=0.90)
    sample_counts = [ps.smooth[k].n_samples for k, *_ in avail]
    sample_note = (f"n = {sample_counts[0]:,} samples per metric"
                   if len(set(sample_counts)) == 1
                   else "Samples: " + ", ".join(
                       f"{lab} n={ps.smooth[k].n_samples:,}" for k, lab, *_ in avail))
    fig.text(0.5, 0.90, sample_note,
             ha="center", va="top", fontsize=9, color="0.30")
    fig.legend(handles=[plt.Line2D([], [], color="0.08", lw=1.8, label="median"),
                        Patch(facecolor="0.5", alpha=0.16, edgecolor="none",
                              label="central 90% (P5-P95)")],
               loc="lower center", ncol=2, fontsize=8.7, bbox_to_anchor=(0.5, 0.01))
    fig.subplots_adjust(bottom=0.18, wspace=0.12, top=0.78)
    ex.figure(fig, "7_motion_smoothness_distribution", subdir=ps.name)

    df = pd.DataFrame([{"Metric": f"{lab} [{units[k]}]",
                        "Mean": round(ps.smooth[k].mean, 5),
                        "Std": round(ps.smooth[k].std, 5),
                        "95% CI": ps.smooth[k].ci_str(5),
                        "Median": round(ps.smooth[k].median, 5),
                        "Samples": ps.smooth[k].n_samples}
                       for k, lab, unit, _, _ in avail])
    ex.table(df, "7_motion_smoothness_stats", subdir=ps.name,
             caption=f"Motion smoothness statistics - {ps.pretty} "
                     f"(n={ps.n_episodes} episodes, t >= {SMOOTHNESS_START_S:g} s)")
    return df

### numerical tables across every preset

In [9]:
def _r(x, p=2):
    try:
        return round(float(x), p) if x is not None and np.isfinite(float(x)) else float("nan")
    except (TypeError, ValueError):
        return float("nan")


def table_1(presets: Sequence[Preset]) -> pd.DataFrame:
    return pd.DataFrame([{
        "Task Preset": p.pretty, "Instruction": p.instruction, "Episodes": p.n_episodes,
        "Success Rate (%)": _r(p.success_rate, 1),
        "Mean Completion Steps": _r(p.steps.mean, 1), "Std Completion Steps": _r(p.steps.std, 1),
        "Mean Completion Time (s)": _r(p.time.mean, 2), "Std Completion Time (s)": _r(p.time.std, 2),
    } for p in presets])


def table_2(presets: Sequence[Preset]) -> pd.DataFrame:
    return pd.DataFrame([{
        "Task": p.pretty, "Episodes": p.n_episodes,
        "IK Success (%)": _r(p.ik_success.mean, 2), "Std": _r(p.ik_success.std, 2),
        "Full Pose (%)": _r(p.ik_stage["full_pose"].mean, 2),
        "Relaxed Orientation (%)": _r(p.ik_stage["relaxed_orientation"].mean, 2),
        "Nearest Feasible (%)": _r(p.ik_stage["nearest_feasible"].mean, 2),
        "Hold Current (%)": _r(p.ik_stage["hold_current"].mean, 2),
    } for p in presets])


def table_3(presets: Sequence[Preset]) -> pd.DataFrame:
    return pd.DataFrame([{
        "Task": p.pretty,
        "Mean Position Error (mm)": _r(p.ik_pos_err.mean, 3), "Std (mm)": _r(p.ik_pos_err.std, 3),
        "95% CI (mm)": p.ik_pos_err.ci_str(3),
        "Mean Orientation Error (deg)": _r(p.ik_rot_err.mean, 3),
        "Std (deg)": _r(p.ik_rot_err.std, 3), "95% CI (deg)": p.ik_rot_err.ci_str(3),
    } for p in presets])


def table_4(presets: Sequence[Preset]) -> pd.DataFrame:
    return pd.DataFrame([{
        "Task": p.pretty,
        "VLA Latency Mean (ms)": _r(p.lat["vla"].mean, 2), "VLA Std (ms)": _r(p.lat["vla"].std, 2),
        "IK Latency Mean (ms)": _r(p.lat["ik"].mean, 2), "IK Std (ms)": _r(p.lat["ik"].std, 2),
        "Execution Latency Mean (ms)": _r(p.lat["exec"].mean, 2),
        "Execution Std (ms)": _r(p.lat["exec"].std, 2),
    } for p in presets])


def table_5(presets: Sequence[Preset]) -> pd.DataFrame:
    return pd.DataFrame([{
        "Task": p.pretty,
        "Velocity Mean (m/s)": _r(p.smooth["vel"].mean, 4),
        "Velocity Std (m/s)": _r(p.smooth["vel"].std, 4),
        "Acceleration Mean (m/s2)": _r(p.smooth["acc"].mean, 4),
        "Acceleration Std (m/s2)": _r(p.smooth["acc"].std, 4),
        "Jerk Mean (m/s3)": _r(p.smooth["jerk"].mean, 3),
        "Jerk Std (m/s3)": _r(p.smooth["jerk"].std, 3),
    } for p in presets])


def table_6(presets: Sequence[Preset]) -> pd.DataFrame:
    rows = []
    for p in presets:
        eps = [e for e in p.episodes if e.has_collision_data]
        n = len(eps)
        hard_ep = sum(1 for e in eps if e.hard_collision_ticks > 0)
        warn_ep = sum(1 for e in eps if e.warn_collision_ticks > 0)
        clear = [e.min_clearance_m for e in eps if np.isfinite(e.min_clearance_m)]
        cst = episode_stat(clear) if clear else Stat()
        rows.append({
            "Task": p.pretty, "Episodes with data": n,
            "Hard Collision Rate (% episodes)": _r(100.0 * hard_ep / n, 1) if n else float("nan"),
            "Warning Collision Rate (% episodes)": _r(100.0 * warn_ep / n, 1) if n else float("nan"),
            "Warning threshold (mm)": round(WARN_CLEARANCE_M * 1000, 1),
            "Minimum Clearance (mm)": _r(min(clear) * 1000, 2) if clear else float("nan"),
            "Mean Minimum Clearance +/- Std (mm)": (
                f"{cst.mean * 1000:.2f} +/- {cst.std * 1000:.2f}"
                if clear and np.isfinite(cst.std) else (f"{cst.mean * 1000:.2f}" if clear else "n/a")),
        })
    return pd.DataFrame(rows)


C_TABLES = [
    ("1_task_success_evaluation", "Table 1 - Task success evaluation", table_1),
    ("2_ik_performance_summary", "Table 2 - IK performance summary", table_2),
    ("3_ik_accuracy_summary", "Table 3 - IK accuracy summary", table_3),
    ("4_computational_performance", "Table 4 - Computational performance", table_4),
    ("5_motion_quality_summary", "Table 5 - Motion quality summary", table_5),
    ("6_safety_collision_metrics", "Table 6 - Safety and collision metrics", table_6),
]


def _m(a, p=4):
    return _r(np.nanmean(a), p) if a is not None and len(np.asarray(a).ravel()) else float("nan")


def per_episode_frame(presets: Sequence[Preset]) -> pd.DataFrame:
    rows = []
    for p in presets:
        for e in p.episodes:
            row = {"task": p.name, "episode": e.stem, "success": e.success,
                   "stop_reason": e.stop_reason, "vla_steps": e.n_vla_steps, "ticks": e.n_ticks,
                   "duration_sim_s [s]": _r(e.duration_sim_s, 3),
                   "duration_wall_s [s]": _r(e.duration_wall_s, 3),
                   "ik_success [%]": _r(e.ik_success_pct, 2)}
            row.update({f"ik_{k} [%]": _r(e.ik_pct.get(k, np.nan), 2) for k in IK_ALL_KEYS})
            row.update({"ik_pos_err_mean [mm]": _m(e.ik_pos_err_mm),
                        "ik_rot_err_mean [deg]": _m(e.ik_rot_err_deg),
                        "lat_vla_mean [ms]": _m(e.lat_vla_ms, 3),
                        "lat_ik_mean [ms]": _m(e.lat_ik_ms, 3),
                        "lat_exec_mean [ms]": _m(e.lat_exec_ms, 3),
                        "track_pos_err_mean [mm]": _m(e.track_pos_err_mm),
                        "track_rot_err_mean [deg]": _m(e.track_rot_err_deg),
                        "vel_mean [m/s]": _m(e.vel, 5), "acc_mean [m/s2]": _m(e.acc, 5),
                        "jerk_mean [m/s3]": _m(e.jerk, 4),
                        "hard_collision_ticks": e.hard_collision_ticks,
                        "warn_collision_ticks": e.warn_collision_ticks,
                        "min_clearance [mm]": _r(e.min_clearance_m * 1000, 3)
                        if np.isfinite(e.min_clearance_m) else float("nan")})
            rows.append(row)
    return pd.DataFrame(rows)

### Input resolution & driver

In [10]:
def resolve_presets(inputs: Sequence[Any]) -> list:
    """Map each input onto (preset_dir, [.h5 files]).

    * directory holding .h5 files -> one preset
    * directory of preset subdirs -> one preset per subdirectory
    * glob / explicit file list   -> grouped by parent directory
    """
    out: dict = {}
    files: list = []
    for raw in inputs:
        s = str(raw)
        if any(c in s for c in "*?["):
            files += [Path(x) for x in glob.glob(s)]
            continue
        p = Path(s)
        if p.is_dir():
            direct = sorted(p.glob("*.h5"))
            if direct:
                out.setdefault(p, []).extend(direct)
            else:
                for sub in sorted(x for x in p.iterdir() if x.is_dir()):
                    sub_files = sorted(sub.glob("*.h5"))
                    if sub_files:
                        out.setdefault(sub, []).extend(sub_files)
        elif p.is_file():
            files.append(p)
        else:
            print(f"[WARN] input not found: {s}")
    for f in files:
        if f.suffix == ".h5":
            out.setdefault(f.parent, []).append(f)
    return [(d, sorted(set(v))) for d, v in out.items() if v]


def run_evaluation(inputs: Sequence[Any], out_root: Any = "thesis_evaluation", dpi: int = 320,
                   formats: Sequence[str] = ("png", "pdf"), only: Sequence[str] = None):
    plt.rcParams.update(PAPER_RC)
    groups = resolve_presets(inputs)
    if not groups:
        raise FileNotFoundError(f"no .h5 episodes found under {[str(i) for i in inputs]}")
    ex = Exporter(Path(out_root), dpi=dpi, formats=formats)
    wanted = {s.upper() for s in only} if only else None

    print(f"\nSURENA thesis evaluation - {len(groups)} task preset(s) -> {ex.root}")
    presets = []
    for directory, paths in groups:
        print(f"\n[preset] {directory}  ({len(paths)} episodes)")
        ps = build_preset(directory, paths)
        presets.append(ps)
        for tag, fn in (("1", fig_1_ik_donut), ("2", fig_2_latency), ("3", fig_3_pos_error),
                        ("4", fig_4_rot_error), ("5", fig_5_eef_tracking),
                        ("6", fig_6_joint_tracking), ("7", fig_7_smoothness)):
            if wanted and tag not in wanted:
                continue
            try:
                fn(ps, ex)
            except Exception as exc:
                print(f"  [ERROR] {tag} failed: {type(exc).__name__}: {exc}")

    print("\n[tables] Group C")
    tables = {}
    for name, caption, fn in C_TABLES:
        if wanted and name.split("_")[0] not in wanted:
            continue
        df = fn(presets)
        tables[name] = df
        ex.table(df, name, caption=caption)
    ex.table(per_episode_frame(presets), "per_episode_metrics",
             caption="Per-episode metrics (raw batch data behind every table and figure)")

    summary = {
        "presets": [{
            "name": p.name, "instruction": p.instruction, "directory": p.directory,
            "episodes": p.n_episodes, "successes": p.n_success, "scored_episodes": p.n_scored,
            "success_rate_pct": p.success_rate, "success_rate_ci95_pct": p.success_ci,
            "representative_episode": p.representative.stem if p.representative else None,
            "completion_steps": p.steps.as_dict(), "completion_time_s": p.time.as_dict(),
            "ik_success_pct": p.ik_success.as_dict(),
            "ik_stage_pct": {k: v.as_dict() for k, v in p.ik_stage.items()},
            "ik_pos_err_mm": p.ik_pos_err.as_dict(), "ik_rot_err_deg": p.ik_rot_err.as_dict(),
            "latency_ms": {k: v.as_dict() for k, v in p.lat.items()},
            "tracking_pos_err_mm": p.track_pos.as_dict(),
            "tracking_rot_err_deg": p.track_rot.as_dict(),
            "smoothness": {k: v.as_dict() for k, v in p.smooth.items()},
            "raw_ik_stage_names": sorted({k for e in p.episodes for k in e.raw_stage_counts}),
        } for p in presets],
        "conventions": {
            "figures": "batch statistics over every episode of the preset; B5/B6 use one "
                       "representative successful episode (medoid of tracking error)",
            "tables": "mean +/- std across episodes; 95% CI = Student t, df = n_episodes - 1",
            "smoothness_start_s": SMOOTHNESS_START_S,
            "warning_clearance_m": WARN_CLEARANCE_M,
        },
    }
    (ex.root / "evaluation_summary.json").write_text(json.dumps(summary, indent=2, default=str),
                                                     encoding="utf-8")
    print(f"\n[json]  {ex.root / 'evaluation_summary.json'}")
    print(f"done - {len(ex.written)} artifacts in {ex.root}\n")
    return presets, tables

## 3. configure inputs and run the batch evaluation

In [11]:
from pathlib import Path
import pandas as pd
import os
from IPython.display import Image, display, Markdown
from surena_vla.paths import OUTPUTS_ROOT

# input: one preset dir, several preset dirs, or the parent of all presets  
RUNS = Path("/media/parsa/OS/Users/parsa/Desktop/vla/FINAL_RUNS")
PRESETS = [RUNS]                     # e.g. [RUNS / "close_microwave", RUNS / "close_top_drawer"]
OUT_ROOT  = Path(os.environ.get("SURENA_EVALUATION_RESULTS_DIR", OUTPUTS_ROOT / "evaluation_results"))

# optional knobs  
SMOOTHNESS_START_S = 1.0          # startup transient discarded from smoothness metrics
WARN_CLEARANCE_M   = 0.02         # clearance below this counts as a warning collision (Table C6)
print([str(p) for p in PRESETS])

['/media/parsa/OS/Users/parsa/Desktop/vla/FINAL_RUNS']


Every figure and every table below is computed over **all** episodes of each preset.
Figures 5 & 6 use one representative successful episode (medoid of tracking error) because trajectories of different manipulations cannot be meaningfully averaged.

In [12]:
presets, tables = run_evaluation(PRESETS, out_root=OUT_ROOT, dpi=320, formats=("png", "pdf"))


SURENA thesis evaluation - 4 task preset(s) -> /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evaluation_results

[preset] /media/parsa/OS/Users/parsa/Desktop/vla/FINAL_RUNS/close_microwave  (10 episodes)
  preset 'close_microwave': 10 episodes, success 40% (4/10), representative = close_microwave__cd8d74bfc4b5
  [fig]   /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evaluation_results/figures/close_microwave/1_ik_solution_strategy_distribution.png|pdf
  [table] /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evaluation_results/tables/close_microwave/1_ik_solution_strategy_stats.csv|.md|.tex
  [fig]   /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evaluation_results/figures/close_microwave/2_computational_latency_distribution.png|pdf
  [table] /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evaluation_results/tables/close_microwave/2_latency_stats.csv|.md|.tex
  [fig]   /home/parsa/surena-vla-ws/surena-vla-manipulation/outputs/evalu

## 4. Video from the episode log

Frames are decoded directly from each episode HDF5 file; **no re-simulation** is performed.

Two MP4 files are saved for every episode under `OUT_ROOT / "videos"`:

- **Mode A — plain episode:** `<stem>.mp4` at 60 fps from `video_frames`.
- **Mode B — frame + action history:** `<stem>__history.mp4` at 8 fps, using `keyframes` when available (otherwise `video_frames`) and `vla_steps/exec_action` with `raw_action` only as fallback.


In [ ]:
from matplotlib import animation
from surena_vla.telemetry.codec import decode_jpeg

VIDEO_DIR = OUT_ROOT / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

PLAIN_VIDEO_FPS = 60
HISTORY_VIDEO_FPS = 8

_VIDEO_ACTION_LABELS = ["dx", "dy", "dz", "droll", "dpitch", "dyaw", "gripper"]
_VIDEO_ACTION_COLORS = [C_BLUE, C_RED, C_GREEN, C_YELLOW, C_CYAN, C_PURPLE, C_GREY]


def decode_frames_from_h5(path):
    """Decode logged JPEG frames and the executed 7-D VLA action history."""
    path = Path(path)
    with h5py.File(path, "r") as f:
        out = {"keyframes": [], "video_frames": [], "actions": None, "action_source": None}
        if "keyframes" in f:
            out["keyframes"] = [decode_jpeg(b) for b in f["keyframes"][()]]
        if "video_frames" in f:
            out["video_frames"] = [decode_jpeg(b) for b in f["video_frames"][()]]

        task_value = f.attrs.get("instruction", f.attrs.get("preset_name", path.stem))
        out["task"] = _txt(task_value)

        if "vla_steps/exec_action" in f:
            out["actions"] = np.asarray(f["vla_steps/exec_action"][()])
            out["action_source"] = "vla_steps/exec_action"
        elif "vla_steps/raw_action" in f:
            out["actions"] = np.asarray(f["vla_steps/raw_action"][()])
            out["action_source"] = "vla_steps/raw_action"
    return out


def save_mp4(frames, out_path, fps=PLAIN_VIDEO_FPS):
    """Mode A: save the raw logged episode frames as a plain MP4."""
    import imageio.v2 as imageio

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(
        str(out_path),
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    ) as writer:
        for frame in frames:
            frame = np.asarray(frame)
            if frame.dtype != np.uint8:
                frame = np.clip(frame, 0, 255).astype(np.uint8)
            writer.append_data(frame)

    print(f"[video/plain]   {out_path} ({len(frames)} frames @ {fps} fps)")
    return out_path


def _history_animation_setup(frames, actions, task, figsize=(12.5, 5.2)):
    """Build the shared frame + growing 7-D action-history animation."""
    acts = np.asarray(actions, float)[:, :7]
    n = len(acts)
    if n == 0:
        raise ValueError("cannot build action-history video with zero actions")

    frames = list(frames)
    if len(frames) != n and len(frames) > n:
        frames = [frames[i] for i in np.linspace(0, len(frames) - 1, n, dtype=int)]
    frames = list(frames[:n]) or [np.zeros((224, 224, 3), np.uint8)] * n
    if len(frames) < n:
        frames += [frames[-1]] * (n - len(frames))

    fig, (ax_frame, ax_action) = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(f"Instruction: {task}", fontsize=12, fontweight="semibold", y=0.98, wrap=True)

    image_artist = ax_frame.imshow(frames[0])
    ax_frame.set_title("Episode", fontsize=10)
    ax_frame.axis("off")
    step_text = ax_frame.text(
        5, 15, "", color="white", fontsize=9,
        bbox=dict(facecolor="black", alpha=0.55)
    )

    lines = [
        ax_action.plot([], [], label=label, color=color, lw=1.4)[0]
        for label, color in zip(_VIDEO_ACTION_LABELS, _VIDEO_ACTION_COLORS)
    ]

    lo, hi = float(np.nanmin(acts)), float(np.nanmax(acts))
    pad = max(0.1, 0.05 * (hi - lo + 1e-9))
    ax_action.set_xlim(0, max(1, n - 1))
    ax_action.set_ylim(lo - pad, hi + pad)
    ax_action.set_title("Action history", fontsize=10)
    ax_action.set_xlabel("VLA step [count]", fontsize=9)
    ax_action.set_ylabel("Normalized action [1]", fontsize=9)
    ax_action.tick_params(labelsize=8)
    ax_action.axhline(0, color="black", lw=0.6, ls="--", alpha=0.6)
    ax_action.legend(fontsize=7, ncol=2, loc="upper right")

    def update(k):
        image_artist.set_data(frames[k])
        step_text.set_text(f"Step {k + 1}/{n}")
        x = np.arange(k + 1)
        for j, line in enumerate(lines):
            line.set_data(x, acts[:k + 1, j])
        return [image_artist, step_text] + lines

    fig.tight_layout()
    return fig, update, n


def save_action_history_mp4(
    frames, actions, out_path, task="", fps=HISTORY_VIDEO_FPS
):
    """Mode B: save frame + growing VLA action history as MP4; never display inline."""
    fig, update, n = _history_animation_setup(frames, actions, task)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    writer = animation.FFMpegWriter(
        fps=fps,
        metadata={"artist": "surena-vla"},
    )
    with writer.saving(fig, str(out_path), dpi=120):
        for k in range(n):
            update(k)
            writer.grab_frame()

    plt.close(fig)
    print(f"[video/history] {out_path} ({n} VLA steps @ {fps} fps)")
    return out_path


def export_episode_videos(path):
    """Save both video modes for one episode whenever the required logged data exist."""
    path = Path(path)
    data = decode_frames_from_h5(path)
    result = {"plain": None, "history": None}

    if data["video_frames"]:
        result["plain"] = save_mp4(
            data["video_frames"],
            VIDEO_DIR / f"{path.stem}.mp4",
            fps=PLAIN_VIDEO_FPS,
        )
    else:
        print(f"[WARN] {path.name}: no video_frames; plain MP4 not written")

    actions = data.get("actions")
    history_frames = data["keyframes"] or data["video_frames"]
    if actions is not None and len(actions) and history_frames:
        result["history"] = save_action_history_mp4(
            history_frames,
            actions,
            VIDEO_DIR / f"{path.stem}__history.mp4",
            task=data["task"],
            fps=HISTORY_VIDEO_FPS,
        )
    else:
        print(
            f"[WARN] {path.name}: missing action history or frames; "
            "history MP4 not written"
        )

    return result


# Export every episode included in the thesis evaluation.
# video_exports = {}
# for preset in presets:
#     for episode in preset.episodes:
#         path = Path(episode.path)
#         try:
#             video_exports[str(path)] = export_episode_videos(path)
#         except Exception as exc:
#             print(f"[ERROR] video export failed for {path.name}: {type(exc).__name__}: {exc}")

# n_plain = sum(v.get("plain") is not None for v in video_exports.values())
# n_history = sum(v.get("history") is not None for v in video_exports.values())
# print(
#     f"video export complete -> {VIDEO_DIR} | "
#     f"plain={n_plain}/{len(video_exports)}, history={n_history}/{len(video_exports)}"
# )


## 5. In-place Display

### numerical tables

In [ ]:
for name, caption, _ in C_TABLES:
    if name in tables:
        display(Markdown(f"### {caption}"))
        display(tables[name])

### Figures

In [ ]:
FIGS = ["1_ik_solution_strategy_distribution", "2_computational_latency_distribution",
        "3_ik_position_error_distribution", "4_ik_orientation_error_distribution",
        "5_eef_trajectory_tracking", "6_joint_trajectory_tracking",
        "7_motion_smoothness_distribution"]

for ps in presets:
    display(Markdown(f"### {ps.pretty} (n={ps.n_episodes} episodes)"))
    for f in FIGS:
        png = OUT_ROOT / "figures" / ps.name / f"{f}.png"
        if png.exists():
            display(Image(filename=str(png), width=760))

### Per-figure statistics tables

Any value shown as mean ± std in a figure is also written next to it as a table (`tables/<preset>/*.csv|md|tex`).

In [ ]:
for ps in presets:
    display(Markdown(f"### {ps.pretty}"))
    for csv in sorted((OUT_ROOT / "tables" / ps.name).glob("B*.csv")):
        display(Markdown(f"**{csv.stem}**"))
        display(pd.read_csv(csv))